# Datos y tensores

**Capítulo 1 · Universidad de las Hespérides**

Adaptación al español de *Dive into Deep Learning*, Aston Zhang, Zachary C. Lipton, Mu Li y Alexander J. Smola.
Fuente: `locked/chapter_preliminaries/ndarray.ipynb` · [Lección original](https://d2l.ai/chapter_preliminaries/ndarray.html).
Texto adaptado bajo [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). [Procedencia y cambios](../PROCEDENCIA.md).
Se conserva la secuencia de las celdas y de los ejercicios; las notas de Hespérides se identifican expresamente.

**Entorno:** ejecuta `uv sync` en la raíz y selecciona su Python como kernel. Las descargas se realizan una vez y quedan en `data/`.
Por defecto, el soporte limita los entrenamientos de `Trainer` a tres épocas y 1024/256 ejemplos para CPU.
Para repetir el régimen completo, inicia Jupyter con `HESPERIDES_COMPLETO=1`. Los ejemplos visuales pequeños conservan su propia configuración explícita.
Los datos de texto en inglés o francés son entradas de los experimentos originales y mantienen su idioma.


In [ ]:
from pathlib import Path
import sys
RAIZ = Path.cwd() if (Path.cwd() / "laboratorio").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from laboratorio import d2l, configurar, epocas
configurar()


# Manipulación de datos
<a id="sec_ndarray"></a>

Para trabajar con datos necesitamos adquirirlos, almacenarlos y procesarlos. Empecemos por los arrays de $n$ dimensiones, a los que también llamamos *tensores*. Si ya conoces NumPy, muchas de estas operaciones te resultarán familiares. Las clases de tensores de las bibliotecas de Deep Learning —`ndarray` en MXNet y `Tensor` en PyTorch y TensorFlow— se parecen al `ndarray` de NumPy, pero incorporan dos capacidades especialmente útiles: diferenciación automática y cómputo en GPU. NumPy, en su implementación habitual, opera en CPU. Estas herramientas permiten expresar redes neuronales con poco código y ejecutar sus operaciones numéricas eficientemente.

## Primeros pasos

**Para empezar, importamos la biblioteca PyTorch. Tenga en cuenta que el nombre del paquete es `torch`.**


In [ ]:
import torch

**Un tensor representa una matriz (posiblemente multidimensional) de valores numéricos.** En el caso unidimensional, es decir, cuando sólo se necesita un eje para los datos, un tensor se llama un *vector*. Con dos ejes, un tensor se llama una *matriz*. Con ejes $k > 2$, soltamos los nombres especializados y simplemente nos referimos al objeto como un tensor $k^\textrm{th}$-*orden*.


PyTorch proporciona una variedad de funciones para crear nuevos tensores prepoblados con valores. Por ejemplo, invocando `arange(n)`, podemos crear un vector de valores espaciados uniformemente, comenzando en 0 (incluido) y terminando en `n` (no incluido). Por defecto, el tamaño del intervalo es $1$. A menos que se especifique lo contrario, los nuevos tensores se almacenan en la memoria principal y se designan para el cálculo basado en CPU.


In [ ]:
x = torch.arange(12, dtype=torch.float32)
x

Cada uno de estos valores se llama *elemento* del tensor. El tensor `x` contiene 12 elementos. Podemos inspeccionar el número total de elementos en un tensor a través de su método `numel`.


In [ ]:
x.numel()

** Podemos acceder a *forma de un tensor*** (la longitud a lo largo de cada eje) inspeccionando su atributo `shape`. Debido a que estamos tratando con un vector aquí, el `shape` contiene sólo un elemento y es idéntico al tamaño.


In [ ]:
x.shape

Podemos **cambiar la forma de un tensor sin alterar su tamaño o valores**, invocando `reshape`. Por ejemplo, podemos transformar nuestro vector `x` cuya forma es (12,) a una matriz `X` con forma (3, 4). Este nuevo tensor retiene todos los elementos pero los reconfigura en una matriz. Observe que los elementos de nuestro vector se establecen una fila a la vez y por lo tanto `x[3] == X[0, 3]`.


In [ ]:
X = x.reshape(3, 4)
X

Tenga en cuenta que especificar cada componente de forma a `reshape` es redundante. Debido a que ya sabemos el tamaño de nuestro tensor, podemos elaborar un componente de la forma dado el resto. Por ejemplo, dado un tensor de tamaño $n$ y forma de destino ($h$, $w$), sabemos que $w = n/h$. Para inferir automáticamente un componente de la forma, podemos colocar un `-1` para el componente de forma que debe inferirse automáticamente. En nuestro caso, en lugar de llamar a `x.reshape(3, 4)`, podríamos haber llamado equivalentemente `x.reshape(-1, 4)` o `x.reshape(3, -1)`.

Los practicantes a menudo necesitan trabajar con tensores inicializados para contener todos los 0s o 1s. **Podemos construir un tensor con todos los elementos ajustados a 0** ( o ) y una forma de (2, 3, 4) a través de la función `zeros`.


In [ ]:
torch.zeros((2, 3, 4))

Del mismo modo, podemos crear un tensor con todos 1s invocando `ones`.


In [ ]:
torch.ones((2, 3, 4))

A menudo deseamos **muestrar cada elemento al azar (e independientemente)** de una distribución de probabilidad dada. Por ejemplo, los parámetros de las redes neuronales a menudo se inicializan al azar. El siguiente fragmento crea un tensor con elementos extraídos de una distribución estándar de Gaussian (normal) con media 0 y desviación estándar 1.


In [ ]:
torch.randn(3, 4)

Finalmente, podemos construir tensores mediante **suministrando los valores exactos para cada elemento** suministrando (posiblemente anidados) listas de Python que contienen literales numéricos. Aquí, construimos una matriz con una lista de listas, donde la lista externa corresponde al eje 0, y la lista interna corresponde al eje 1.


In [ ]:
torch.tensor([[2, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]])

## Indexación y selección de segmentos
Al igual que con las listas de Python, podemos acceder a elementos tensores mediante indexación (comenzando con 0). Para acceder a un elemento basado en su posición relativa al final de la lista, podemos utilizar la indexación negativa. Finalmente, podemos acceder a rangos completos de índices a través de cortes (por ejemplo, `X[start:stop]`), donde el valor devuelto incluye el primer índice (`start`) *pero no el último* (`stop`). Finalmente, cuando se especifica un solo índice (o corte) para un tensor de orden $k^\textrm{th}$, se aplica a lo largo del eje 0. Así, en el siguiente código, **`[-1]` selecciona la última fila y `[1:3]` selecciona la segunda y tercera filas**.


In [ ]:
X[-1], X[1:3]

Más allá de leerlos, ** también podemos *escribir* elementos de una matriz especificando índices.**


In [ ]:
X[1, 2] = 17
X

Si queremos ** asignar múltiples elementos del mismo valor, aplicamos la indexación en el lado izquierdo de la operación de asignación.** Por ejemplo, `[:2, :]` accede a la primera y segunda filas, donde `:` toma todos los elementos a lo largo del eje 1 (columna).Mientras discutimos la indexación para matrices, esto también funciona para vectores y para tensores de más de dos dimensiones.


In [ ]:
X[:2, :] = 12
X

## Operaciones
Ahora que sabemos cómo construir tensores y cómo leer y escribir a sus elementos, podemos empezar a manipularlos con varias operaciones matemáticas. Entre las más útiles de estas son las operaciones *elementwise*. Estas aplican una operación escalar estándar a cada elemento de un tensor. Para las funciones que toman dos tensores como entradas, las operaciones elementowise aplican algún operador binario estándar en cada par de elementos correspondientes. Podemos crear una función elementowise de cualquier función que mapee desde un escalar a un escalar.

En la notación matemática, denotamos tales operadores escalares *unarios* (tomando una entrada) por la firma $f: \mathbb{R} \rightarrow \mathbb{R}$. Esto sólo significa que la función mapea desde cualquier número real a algún otro número real. La mayoría de los operadores estándar, incluyendo los unarios como $e^x$, se pueden aplicar como elemento.


### Nota docente de Hespérides

Una forma correcta no garantiza un significado correcto: fija qué representa cada eje antes de calcular. En el primer entrenamiento, distingue logits, probabilidades y etiquetas. La ilustración presenta una intuición; el explorador final muestra coordenadas realmente calculadas por una red pequeña.

Vínculo con los apuntes: sesión 1, «Datos y tensores».


In [ ]:
torch.exp(x)

Asimismo, denotamos a los operadores escalares *binarios*, que mapean pares de números reales a un número real (único) a través de la firma $f: \mathbb{R}, \mathbb{R} \rightarrow \mathbb{R}$. Dados los dos vectores $\mathbf{u}$ y $\mathbf{v}$ *de la misma forma*, y un operador binario $f$, podemos producir un vector $\mathbf{c} = F(\mathbf{u},\mathbf{v})$ configurando $c_i \gets f(u_i, v_i)$ para todo $i$, donde $c_i, u_i$, y $v_i$ son los elementos $i^\textrm{th}$ de los vectores $\mathbf{c}, \mathbf{u}$, y $\mathbf{v}$. Aquí, produjimos el vector valorado $F: \mathbb{R}^d, \mathbb{R}^d \rightarrow \mathbb{R}^d$ mediante *elevando* la función escalar a una operación vectorial en sentido de elemento. Los operadores aritméticas estándar comunes para la adición (`+`), resta (`-`), multiplicación (`*`), división (`/`), y exponenciación (`**`) han sido todos *elevados* a operaciones elementales para tensores de forma arbitraria de forma idéntica.


In [ ]:
x = torch.tensor([1.0, 2, 4, 8])
y = torch.tensor([2, 2, 2, 2])
x + y, x - y, x * y, x / y, x ** y

Además de los cálculos de elementos, también podemos realizar operaciones algebraicas lineales, tales como productos de puntos y multiplicaciones de matrices. [Referencia sec_linear-algebra](https://d2l.ai/chapter_preliminaries/linear-algebra.html#sec-linear-algebra).

También podemos ***concatenar* varios tensores,** apilarlos de extremo a extremo para formar uno más grande. Sólo necesitamos proporcionar una lista de tensores y decirle al sistema a lo largo de qué eje concatenar. El ejemplo siguiente muestra lo que sucede cuando concatenamos dos matrices a lo largo de filas (eje 0) en lugar de columnas (eje 1). Podemos ver que la primera salida eje-0 longitud ($6$) es la suma de los dos tensores de entrada eje-0 longitudes ($3 + 3$); mientras que la segunda salida eje-1 longitud ($8$) es la suma de los dos tensores de entrada eje-1 longitudes ($4 + 4$).


In [ ]:
X = torch.arange(12, dtype=torch.float32).reshape((3,4))
Y = torch.tensor([[2.0, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]])
torch.cat((X, Y), dim=0), torch.cat((X, Y), dim=1)

A veces, queremos **construir un tensor binario a través de *indicaciones lógicas*.** Tome `X == Y` como ejemplo. Para cada posición `i, j`, si `X[i, j]` y `Y[i, j]` son iguales, entonces la entrada correspondiente en el resultado toma el valor `1`, de lo contrario toma el valor `0`.


In [ ]:
X == Y

** Sumar todos los elementos en el tensor** produce un tensor con un solo elemento.


In [ ]:
X.sum()

## Broadcasting
<a id="subsec_broadcasting"></a>

Por ahora, usted sabe cómo realizar operaciones binarias en función de los elementos en dos tensores de la misma forma. Bajo ciertas condiciones, incluso cuando las formas difieren, todavía podemos ** realizar operaciones binarias en función de los elementos invocando el mecanismo * de broadcasting*.** La broadcasting funciona de acuerdo con el siguiente procedimiento de dos pasos: (i) ampliar uno o ambos arrays copiando elementos a lo largo de ejes con longitud 1 para que después de esta transformación, los dos tensores tengan la misma forma; (ii) realizar una operación en función de los elementos en los arrays resultantes.


In [ ]:
a = torch.arange(3).reshape((3, 1))
b = torch.arange(2).reshape((1, 2))
a, b

Dado que `a` y `b` son matrices $3\times1$ y $1\times2$, respectivamente, sus formas no coinciden. La transmisión produce una matriz $3\times2$ más grande replicando la matriz `a` a lo largo de las columnas y la matriz `b` a lo largo de las filas antes de añadirlos en sentido de elementos.


In [ ]:
a + b

## Ahorro de memoria
**Las operaciones de funcionamiento pueden hacer que se asigne nueva memoria a los resultados de host.** Por ejemplo, si escribimos `Y = X + Y`, desmitimos el tensor al que `Y` solía apuntar y en su lugar el punto `Y` en la memoria recientemente asignada. Podemos demostrar este problema con la función `id()` de Python, que nos da la dirección exacta del objeto referenciado en memoria. Tenga en cuenta que después de ejecutar `Y = Y + X`, `id(Y)` apunta a una ubicación diferente. Esto es porque Python evalúa por primera vez `Y + X`, asignando nueva memoria para el resultado y luego puntos `Y` a esta nueva ubicación en memoria.


In [ ]:
before = id(Y)
Y = Y + X
id(Y) == before

Esto podría ser indeseable por dos razones. En primer lugar, no queremos correr alrededor de asignar memoria innecesariamente todo el tiempo. En el aprendizaje automático, a menudo tenemos cientos de megabytes de parámetros y actualizar todos ellos múltiples veces por segundo. Siempre que sea posible, queremos realizar estas actualizaciones *en su lugar*. En segundo lugar, podemos señalar los mismos parámetros de múltiples variables. Si no actualizamos en su lugar, debemos tener cuidado de actualizar todas estas referencias, no sea que se produzca una fuga de memoria o se refiera inadvertidamente a parámetros rancios.


Afortunadamente, ** realizar operaciones en el lugar** es fácil. Podemos asignar el resultado de una operación a un array `Y` previamente asignado usando notación de corte: `Y[:] = <expression>`. Para ilustrar este concepto, sobrescribimos los valores del tensor `Z`, después de inicializarlo, usando `zeros_like`, para tener la misma forma que `Y`.


In [ ]:
Z = torch.zeros_like(Y)
print('id(Z):', id(Z))
Z[:] = X + Y
print('id(Z):', id(Z))

**Si el valor de `X` no es reutilizado en cálculos posteriores, también podemos utilizar `X[:] = X + Y` o `X += Y` para reducir los gastos de memoria de la operación.**


In [ ]:
before = id(X)
X += Y
id(X) == before

## Conversión a otros objetos de Python

**Convertirse en un tensor NumPy (`ndarray`)**, o viceversa, es fácil. El tensor de antorchas y la matriz NumPy compartirán su memoria subyacente, y cambiar uno a través de una operación en el lugar también cambiará el otro.


In [ ]:
A = X.numpy()
B = torch.from_numpy(A)
type(A), type(B)

Para **convertir un tensor de tamaño-1 a un escalar de Python**, podemos invocar la función `item` o las funciones integradas de Python.


In [ ]:
a = torch.tensor([3.5])
a, a.item(), float(a), int(a)

## Resumen
La clase tensor es la interfaz principal para almacenar y manipular datos en bibliotecas de aprendizaje profundo. Los tensores proporcionan una variedad de funcionalidades incluyendo rutinas de construcción; indexación y corte; operaciones matemáticas básicas; broadcasting; asignación de memoria eficiente; y conversión hacia y desde otros objetos de Python.

## Ejercicios
1. Ejecute el código en esta sección. Cambie la instrucción condicional `X == Y` a `X < Y` o `X > Y`, y luego vea qué tipo de tensor puede obtener.
1. Reemplazar los dos tensores que operan por elemento en el mecanismo de broadcasting por otras formas, por ejemplo, tensores tridimensionales. ¿El resultado es el mismo que se esperaba?


[Debate del original](https://discuss.d2l.ai/t/27)
